# 5. Main Regression Analysis: Wealth × Cardiometabolic Burden Interaction

**Research question:** Does cardiometabolic burden (diabetes, hypertension, obesity) modify the
association between household wealth and probable depression / probable anxiety among currently
married women in Bangladesh (BDHS 2022)?

**Design:** Survey-weighted logistic regression, two parallel outcome models (depression,
anxiety), each with a nested Model 1 (main effects) → Model 2 (+ Wealth × Burden interaction)
structure, plus a Model 3 sensitivity check with a coarser (less parametrically restrictive)
coding of the interaction. Sample: n = 4,887 currently married women (identical for both
outcomes) — see `docs/Dictionary.md` and `Figure-2` for the derivation and attrition.

**A note on scope before we start:** this notebook implements the model in Python
(`statsmodels`) as the working/exploratory analysis. Python's survey-design support has two
real limitations relative to R's `survey` package or Stata's `svy:` prefix, both flagged
explicitly below where they matter:

1. `statsmodels` has no native stratified-multistage variance estimator — clustering by PSU is
   supported via cluster-robust ("sandwich") standard errors, but **stratification by `Stratum`
   is not incorporated** into the variance calculation.
2. `statsmodels` itself warns that its cluster-robust covariance is *"not fully supported"* when
   combined with weights — this is a known, disclosed limitation of the library, not something
   specific to this dataset.

To compensate, this notebook (a) uses cluster-robust SEs by PSU as the primary Python-side
estimate, (b) cross-checks them against a **stratified cluster bootstrap** that resamples PSUs
within stratum — which *does* respect the two-stage stratified design — and (c) provides the
exact R `survey::svyglm` / Stata `svy: logit` syntax needed to reproduce the final, submission-
grade standard errors outside Python. Point estimates (the odds ratios themselves) do not depend
on any of this — only the standard errors and p-values do.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import warnings

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

dep = pd.read_csv('../resources/depression_dataset.csv')
anx = pd.read_csv('../resources/anxiety_dataset.csv')

print('Depression analytic sample:', dep.shape)
print('Anxiety analytic sample:   ', anx.shape)
assert dep.shape[0] == anx.shape[0] == 4887, "Sample size drifted from the expected n = 4,887 -- re-run Notebooks 2-4 before proceeding."


Depression analytic sample: (4887, 27)
Anxiety analytic sample:    (4887, 27)


## 2. Outcome definition

`Depression` and `Anxiety` in the analytic datasets are **severity bands**, not binary case
indicators (PHQ-9 for depression, GAD-7 for anxiety; see `docs/Dictionary.md`). The project's
stated analytic framework is survey-weighted **logistic** regression, which requires a binary
outcome, so each is dichotomized at the validated clinical screening threshold:

- **Probable depression** = PHQ-9 ≥ 10, i.e. `Depression` band ∈ {2, 3, 4} (moderate or worse).
  Cutoff per Kroenke, Spitzer & Williams (2001), *J Gen Intern Med*.
- **Probable anxiety** = GAD-7 ≥ 10, i.e. `Anxiety` band ∈ {2, 3} (moderate or worse).
  Cutoff per Spitzer, Kroenke, Williams & Löwe (2006), *Arch Intern Med*.

**This is a documented assumption, not something dictated by the data** — a lower cutoff
(any symptoms, band ≥ 1) or an ordinal/multinomial model on the full severity band would both be
defensible alternatives. Flag this choice for review before treating these results as final.

In [2]:
dep['Depression_binary'] = (dep['Depression'] >= 2).astype(int)
anx['Anxiety_binary'] = (anx['Anxiety'] >= 2).astype(int)

for name, df, col in [('Depression', dep, 'Depression_binary'), ('Anxiety', anx, 'Anxiety_binary')]:
    n_events = df[col].sum()
    unweighted_prev = df[col].mean()
    weighted_prev = np.average(df[col], weights=df['Sampling weight'])
    print(f"{name}: {n_events} probable cases / {df.shape[0]} "
          f"(unweighted prevalence {unweighted_prev:.1%}, weighted {weighted_prev:.1%})")


Depression: 235 probable cases / 4887 (unweighted prevalence 4.8%, weighted 4.8%)
Anxiety: 221 probable cases / 4887 (unweighted prevalence 4.5%, weighted 4.6%)


## 3. Variable coding

All 12 covariates enter as categorical (`C()`), matching exactly how they were coded in the
Notebook 4 GVIF check (confirmed against that notebook's degrees-of-freedom column, so the
multicollinearity diagnostic and the fitted model use the same functional form for every term).
Reference categories are set below; all are substantively natural baselines (youngest age band,
no education, no children, etc.) except **Wealth**, where **Richest** is the reference so that
coefficients for poorer categories are directly interpretable as the hypothesized gradient.

In [3]:
ref_levels = {
    'Socioeconomic Status': [5, 1, 2, 3, 4],   # ref = Richest
    'Education':            [0, 1, 2, 3],       # ref = No education
    'Occupation':           [0, 1],             # ref = No
    'Partner occupation':   [2, 1, 3],           # ref = Working
    'Age':                  [1, 2, 3],           # ref = 15-24
    'Division':             [3, 1, 2, 4, 5, 6, 7, 8],  # ref = Dhaka
    'Residence':            [1, 2],             # ref = Urban
    'Religion':             [1, 2],             # ref = Islam
    'Children':             [0, 1, 2, 3, 4],    # ref = No children
    'Family size':          [1, 2],             # ref = <5 members
    'Household Autonomy':   [0, 1, 2, 3],       # ref = No autonomy
    'Insurance':            [0, 1],             # ref = No
    'Internet':             [0, 1, 2],          # ref = Never
}

covariate_cols = ['Education', 'Occupation', 'Partner occupation', 'Age', 'Division', 'Residence',
                   'Religion', 'Children', 'Family size', 'Household Autonomy', 'Insurance', 'Internet']

def apply_ref_levels(df):
    df = df.copy()
    for col, order in ref_levels.items():
        df[col] = pd.Categorical(df[col], categories=order)
    return df

dep = apply_ref_levels(dep)
anx = apply_ref_levels(anx)
dep['Burden_num'] = dep['Cardiometabolic Burden'].astype(int)
anx['Burden_num'] = anx['Cardiometabolic Burden'].astype(int)
print('Reference levels set for', len(ref_levels), 'variables.')


Reference levels set for 13 variables.


## 4. Checking the Wealth × Burden interaction is estimable

Before fitting any interaction, cross-tabulate Wealth × Burden. A fully categorical 5×4
interaction needs every cell populated with enough events to identify a stable coefficient.

In [4]:
print("Wealth x Cardiometabolic Burden (uncollapsed, 4 levels):")
display(pd.crosstab(dep['Socioeconomic Status'], dep['Cardiometabolic Burden']))


Wealth x Cardiometabolic Burden (uncollapsed, 4 levels):


Cardiometabolic Burden,0,1,2,3
Socioeconomic Status,,,,
5,710,290,108,16
1,687,145,26,1
2,733,175,30,6
3,725,173,40,6
4,716,217,76,7


**`Wealth = Poorest × Burden = 3` has exactly 1 observation.** A fully-categorical
Wealth × Burden interaction (5 × 4 = 20 cells, 12 free interaction parameters) is not reliably
estimable — that one cell alone would produce a coefficient driven by a single person, with an
enormous, uninformative confidence interval.

Two specifications are used instead:

- **Model 2 (primary):** Burden enters as a linear term (0–3). This treats "one more
  cardiometabolic condition" as a constant-effect step on the log-odds scale — a real
  assumption, but one that pools information across all 4,887 women rather than isolating it in
  sparse cells, and it matches how Burden is described in the DAG (`Figure-1`: *"burden score
  (0–3)"*).
- **Model 3 (sensitivity):** Burden collapsed to 3 levels (0 / 1 / 2+), relaxing the linearity
  assumption while keeping every cell above n = 27 (shown below).

In [5]:
dep['Burden_collapsed'] = dep['Cardiometabolic Burden'].astype(int).map({0: 0, 1: 1, 2: 2, 3: 2})
anx['Burden_collapsed'] = anx['Cardiometabolic Burden'].astype(int).map({0: 0, 1: 1, 2: 2, 3: 2})
dep['Burden_collapsed'] = pd.Categorical(dep['Burden_collapsed'], categories=[0, 1, 2])
anx['Burden_collapsed'] = pd.Categorical(anx['Burden_collapsed'], categories=[0, 1, 2])

print("Wealth x Burden_collapsed (0 / 1 / 2+) -- minimum cell size:")
ct = pd.crosstab(dep['Socioeconomic Status'], dep['Burden_collapsed'])
display(ct)
print("Minimum cell count:", ct.values.min())


Wealth x Burden_collapsed (0 / 1 / 2+) -- minimum cell size:


Burden_collapsed,0,1,2
Socioeconomic Status,,,
5,710,290,124
1,687,145,27
2,733,175,36
3,725,173,46
4,716,217,83


Minimum cell count: 27


**A second, separate sparsity check: zero-event covariate categories.** Beyond the Wealth x Burden interaction cells, it's worth checking whether any of the 12 adjustment-set covariates have a category with zero outcome events -- that's complete separation, and the fitted coefficient for that category will be numerically near 0 or near infinity regardless of sample size in the rest of the model, with a meaningless standard error.

In [6]:
def check_zero_event_cells(df, outcome_col, covariates):
    flagged = []
    for v in covariates:
        ct = pd.crosstab(df[v], df[outcome_col])
        zero_rows = ct.index[(ct == 0).any(axis=1)]
        for level in zero_rows:
            flagged.append((v, level, int(ct.loc[level].sum())))
    return flagged

for label, df, outcome in [('Depression', dep, 'Depression_binary'), ('Anxiety', anx, 'Anxiety_binary')]:
    flags = check_zero_event_cells(df, outcome, covariate_cols)
    print(f'{label}: categories with zero outcome events:')
    for v, level, n in flags:
        print(f'  {v} = {level} (n={n}) -- complete separation, coefficient is not reliably estimable')
    if not flags:
        print('  none')


Depression: categories with zero outcome events:
  Partner occupation = 3 (n=8) -- complete separation, coefficient is not reliably estimable
  Insurance = 1 (n=16) -- complete separation, coefficient is not reliably estimable


Anxiety: categories with zero outcome events:


  Insurance = 1 (n=16) -- complete separation, coefficient is not reliably estimable


## 5. Model-fitting helper

A single function fits a survey-weighted logistic model (weights via `var_weights`,
cluster-robust SEs by PSU), checks convergence, and returns a tidy odds-ratio table.

In [7]:
def fit_svy_logit(formula, data, weight_col='Sampling weight', cluster_col='PSU'):
    '''Survey-weighted logistic regression: weighted pseudo-likelihood point estimates,
    cluster-robust (PSU) sandwich standard errors. See the Section 1 caveat re: Stratum and the
    statsmodels SpecificationWarning.'''
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        model = smf.glm(formula=formula, data=data, family=sm.families.Binomial(),
                         var_weights=data[weight_col])
        res = model.fit(cov_type='cluster', cov_kwds={'groups': data[cluster_col]})
        spec_warnings = [str(w.message) for w in caught if issubclass(w.category, sm.tools.sm_exceptions.SpecificationWarning)]
    if not res.converged:
        print("*** WARNING: model did not converge ***")
    return res, spec_warnings

def or_table(res, label=''):
    '''Tidy odds-ratio table with 95% CI from a fitted GLM result.'''
    params = res.params
    ci = res.conf_int()
    out = pd.DataFrame({
        'term': params.index,
        'OR': np.exp(params.values),
        'CI_low': np.exp(ci[0].values),
        'CI_high': np.exp(ci[1].values),
        'p': res.pvalues.values,
    })
    out = out[out['term'] != 'Intercept'].reset_index(drop=True)
    if label:
        out.insert(0, 'model', label)
    return out

def lr_test(res_restricted, res_full):
    '''Likelihood-ratio test for nested models (deviance difference ~ chi-square).'''
    df_diff = res_full.df_model - res_restricted.df_model
    stat = res_restricted.deviance - res_full.deviance
    p = stats.chi2.sf(stat, df_diff)
    return stat, df_diff, p


## 6. Depression models

- **Model 1:** main effects only (Wealth + Burden + 12 covariates)
- **Model 2 (primary):** + Wealth × Burden (linear) interaction
- **Model 3 (sensitivity):** + Wealth × Burden (collapsed 0/1/2+) interaction

In [8]:
covariate_formula = ' + '.join([f'C(Q("{c}"))' for c in covariate_cols])

f_dep_m1  = f'Depression_binary ~ C(Q("Socioeconomic Status")) + Burden_num + {covariate_formula}'
f_dep_m2  = f'Depression_binary ~ C(Q("Socioeconomic Status")) * Burden_num + {covariate_formula}'
# Model 3's main-effect comparator must use the same Burden_collapsed parameterization as Model 3
# itself -- Model 1 (Burden_num) is NOT nested inside Model 3 (Burden_collapsed): they encode the
# moderator differently, not just with/without an interaction. A plain deviance-difference test
# between non-nested models (as an earlier version of this notebook ran) is not statistically
# valid, even though statsmodels does not raise an error for it. R's survey::anova.svyglm does
# check this and refuses ("models not nested") -- see the companion R notebook, Section 6.
f_dep_m1c = f'Depression_binary ~ C(Q("Socioeconomic Status")) + Q("Burden_collapsed") + {covariate_formula}'
f_dep_m3  = f'Depression_binary ~ C(Q("Socioeconomic Status")) * Q("Burden_collapsed") + {covariate_formula}'

res_dep_m1,  w1  = fit_svy_logit(f_dep_m1,  dep)
res_dep_m2,  w2  = fit_svy_logit(f_dep_m2,  dep)
res_dep_m1c, w1c = fit_svy_logit(f_dep_m1c, dep)
res_dep_m3,  w3  = fit_svy_logit(f_dep_m3,  dep)

print('Model 1 converged:', res_dep_m1.converged, '| Model 2 converged:', res_dep_m2.converged, '| Model 3 converged:', res_dep_m3.converged)
if w2:
    print('statsmodels caveat (Model 2):', w2[0])


Model 1 converged: True | Model 2 converged: True | Model 3 converged: True
statsmodels caveat (Model 2): cov_type not fully supported with var_weights


In [9]:
print("=== Depression Model 2 (primary): Wealth x Burden odds ratios ===")
dep_m2_or = or_table(res_dep_m2, 'Depression M2')
display(dep_m2_or.round(3))


=== Depression Model 2 (primary): Wealth x Burden odds ratios ===


,model,term,OR,CI_low,CI_high,p
0,Depression M2,"C(Q(""Socioeconomic Status""))[T.1]",1.597,0.818,3.118,0.170
1,Depression M2,"C(Q(""Socioeconomic Status""))[T.2]",2.042,1.084,3.844,0.027
2,Depression M2,"C(Q(""Socioeconomic Status""))[T.3]",1.787,0.916,3.488,0.089
3,Depression M2,"C(Q(""Socioeconomic Status""))[T.4]",1.647,0.885,3.068,0.116
4,Depression M2,"C(Q(""Education""))[T.1]",0.910,0.597,1.387,0.661
5,Depression M2,"C(Q(""Education""))[T.2]",0.850,0.542,1.335,0.481
6,Depression M2,"C(Q(""Education""))[T.3]",0.473,0.254,0.881,0.018
7,Depression M2,"C(Q(""Occupation""))[T.1]",0.923,0.664,1.282,0.632
8,Depression M2,"C(Q(""Partner occupation""))[T.1]",1.182,0.483,2.890,0.715
9,Depression M2,"C(Q(""Partner occupation""))[T.3]",0.000,0.000,0.000,0.000


**Reading this table:** with an interaction present, the `Socioeconomic Status` main-effect
rows describe the wealth OR **only at `Burden_num = 0`**. The `...:Burden_num` rows are the
*change* in that wealth OR per one-unit increase in burden — not a standalone effect. Section 8
(simple slopes) converts these into the wealth OR at each observed burden level (0–3), which is
what should actually be reported and interpreted.

In [10]:
stat, df_diff, p = lr_test(res_dep_m1, res_dep_m2)
print(f"LR test, Depression Model 1 vs Model 2 (linear interaction): "
      f"chi2({df_diff:.0f}) = {stat:.2f}, p = {p:.4f}")

stat3, df_diff3, p3 = lr_test(res_dep_m1c, res_dep_m3)
print(f"LR test, Depression Model 1c vs Model 3 (collapsed-Burden interaction, correctly nested): "
      f"chi2({df_diff3:.0f}) = {stat3:.2f}, p = {p3:.4f}")


LR test, Depression Model 1 vs Model 2 (linear interaction): chi2(4) = 6.25, p = 0.1816
LR test, Depression Model 1c vs Model 3 (collapsed-Burden interaction, correctly nested): chi2(8) = 18.70, p = 0.0166


**Correction (from cross-checking against the R notebook):** an earlier version of this cell compared Model 1 (Burden as linear) against Model 3 (Burden collapsed to 0/1/2+) directly. That comparison is invalid -- the two models parameterize the moderator differently, so they are not nested, and a deviance-difference test between non-nested models is not statistically meaningful even though `statsmodels` did not raise an error for it. R's `survey::anova.svyglm` correctly refuses this exact comparison ("models not nested"). **Model 1c** above uses the same `Burden_collapsed` main effect as Model 3, making it the valid nested comparator.

With that fix, **Model 3 is still significant where Model 2 is not** (see the LR test below), which suggests the wealth x burden relationship genuinely is not well captured by a straight linear slope across 0-3 burdens -- something specific happens once burden reaches "2 or more" that the linear model washes out. The interaction terms below show where.

In [11]:
dep_m3_or = or_table(res_dep_m3, 'Depression M3')
interaction_rows = dep_m3_or[dep_m3_or['term'].str.contains(':')]
print("Depression Model 3: Wealth x Burden_collapsed interaction terms")
display(interaction_rows.round(3))


Depression Model 3: Wealth x Burden_collapsed interaction terms


,model,term,OR,CI_low,CI_high,p
34,Depression M3,"C(Q(""Socioeconomic Status""))[T.1]:Q(""Burden_co...",1.035,0.278,3.856,0.959
35,Depression M3,"C(Q(""Socioeconomic Status""))[T.2]:Q(""Burden_co...",1.137,0.383,3.381,0.817
36,Depression M3,"C(Q(""Socioeconomic Status""))[T.3]:Q(""Burden_co...",0.683,0.164,2.845,0.600
37,Depression M3,"C(Q(""Socioeconomic Status""))[T.4]:Q(""Burden_co...",2.023,0.564,7.261,0.280
38,Depression M3,"C(Q(""Socioeconomic Status""))[T.1]:Q(""Burden_co...",0.596,0.101,3.501,0.567
39,Depression M3,"C(Q(""Socioeconomic Status""))[T.2]:Q(""Burden_co...",0.146,0.016,1.322,0.087
40,Depression M3,"C(Q(""Socioeconomic Status""))[T.3]:Q(""Burden_co...",0.491,0.100,2.412,0.381
41,Depression M3,"C(Q(""Socioeconomic Status""))[T.4]:Q(""Burden_co...",0.028,0.003,0.250,0.001


### Full Model 2 covariate table (Depression)

In [12]:
with pd.option_context('display.max_rows', 100):
    display(dep_m2_or.round(3))


,model,term,OR,CI_low,CI_high,p
0,Depression M2,"C(Q(""Socioeconomic Status""))[T.1]",1.597,0.818,3.118,0.170
1,Depression M2,"C(Q(""Socioeconomic Status""))[T.2]",2.042,1.084,3.844,0.027
2,Depression M2,"C(Q(""Socioeconomic Status""))[T.3]",1.787,0.916,3.488,0.089
3,Depression M2,"C(Q(""Socioeconomic Status""))[T.4]",1.647,0.885,3.068,0.116
4,Depression M2,"C(Q(""Education""))[T.1]",0.910,0.597,1.387,0.661
5,Depression M2,"C(Q(""Education""))[T.2]",0.850,0.542,1.335,0.481
6,Depression M2,"C(Q(""Education""))[T.3]",0.473,0.254,0.881,0.018
7,Depression M2,"C(Q(""Occupation""))[T.1]",0.923,0.664,1.282,0.632
8,Depression M2,"C(Q(""Partner occupation""))[T.1]",1.182,0.483,2.890,0.715
9,Depression M2,"C(Q(""Partner occupation""))[T.3]",0.000,0.000,0.000,0.000


## 7. Simple slopes: wealth OR at each level of cardiometabolic burden (Depression)

For Wealth level *k* vs. Richest, the log-odds ratio at `Burden_num = b` is
`β_k + b·β_(k:Burden)`. Variance is obtained via the delta method from the model's covariance
matrix, so these are exact linear combinations of Model 2's own parameters — not a separate
model.

In [13]:
def simple_slopes(res, wealth_col_prefix, burden_var, burden_values, wealth_levels):
    cov = res.cov_params()
    params = res.params
    rows = []
    for level in wealth_levels:
        main_term = f'{wealth_col_prefix}[T.{level}]'
        int_term = f'{wealth_col_prefix}[T.{level}]:{burden_var}'
        if main_term not in params.index:
            continue
        for b in burden_values:
            beta = params[main_term] + b * params.get(int_term, 0.0)
            var = cov.loc[main_term, main_term]
            if int_term in cov.index:
                var += (b ** 2) * cov.loc[int_term, int_term]
                var += 2 * b * cov.loc[main_term, int_term]
            se = np.sqrt(var)
            or_ = np.exp(beta)
            lo, hi = np.exp(beta - 1.96 * se), np.exp(beta + 1.96 * se)
            z = beta / se
            p = 2 * (1 - stats.norm.cdf(abs(z)))
            rows.append({'Wealth (vs Richest)': level, 'Burden': b, 'OR': or_, 'CI_low': lo, 'CI_high': hi, 'p': p})
    return pd.DataFrame(rows)

wealth_col = 'C(Q("Socioeconomic Status"))'
dep_slopes = simple_slopes(res_dep_m2, wealth_col, 'Burden_num', [0, 1, 2, 3], [1, 2, 3, 4])
print("Depression: Wealth OR (vs Richest) at each Cardiometabolic Burden level")
display(dep_slopes.round(3))


Depression: Wealth OR (vs Richest) at each Cardiometabolic Burden level


,Wealth (vs Richest),Burden,OR,CI_low,CI_high,p
0,1,0,1.597,0.818,3.118,0.170
1,1,1,1.027,0.464,2.273,0.947
2,1,2,0.660,0.150,2.904,0.583
3,1,3,0.425,0.044,4.093,0.459
4,2,0,2.042,1.084,3.844,0.027
5,2,1,0.993,0.500,1.973,0.984
6,2,2,0.483,0.140,1.667,0.250
7,2,3,0.235,0.035,1.563,0.134
8,3,0,1.787,0.916,3.489,0.089
9,3,1,1.086,0.504,2.339,0.833


## 8. Stratified cluster bootstrap (Depression, Model 2)

Resamples **PSUs with replacement within each `Stratum`**, keeping the number of PSUs per
stratum fixed — this mirrors BDHS's actual two-stage stratified cluster design more closely than
`statsmodels`' plain cluster-robust sandwich (which ignores `Stratum` entirely; see Section 1).
300 replicates (a pragmatic compute-time choice for this environment with the full 38-parameter
model; increase to 1,000+ for a published-table-grade bootstrap if compute time allows).

In [14]:
def stratified_cluster_bootstrap(df, formula, n_boot=300, seed=42, max_abs_coef=15):
    # Fast index-based resampling: precompute (Stratum, PSU) -> row-position array once,
    # then build each replicate by concatenating position arrays (no per-row filtering).
    # Sparse cells (e.g. Insurance=Yes, n=16; Partner occupation=Don't know, n=8) occasionally
    # land at zero count in a resampled replicate, producing a rank-deficient design matrix and
    # a degenerate (quasi-separated) fit. Two guards handle this: (1) skip a replicate up front
    # if its design matrix is rank-deficient, (2) as a backstop, drop any replicate that still
    # produces an implausibly extreme coefficient (|coef| > max_abs_coef, i.e. OR > ~3.3 million).
    df = df.reset_index(drop=True)
    group_indices = df.groupby(['Stratum', 'PSU']).indices
    strata = df['Stratum'].unique()
    psu_by_stratum = {s: df.loc[df['Stratum'] == s, 'PSU'].unique() for s in strata}
    rng = np.random.default_rng(seed)
    boot_params = []
    n_rank_deficient = 0
    n_extreme = 0
    for _ in range(n_boot):
        idx_parts = []
        for s in strata:
            psus = psu_by_stratum[s]
            chosen = rng.choice(psus, size=len(psus), replace=True)
            idx_parts.extend(group_indices[(s, psu)] for psu in chosen)
        boot_df = df.iloc[np.concatenate(idx_parts)].reset_index(drop=True)
        try:
            model = smf.glm(formula=formula, data=boot_df, family=sm.families.Binomial(),
                             var_weights=boot_df['Sampling weight'])
            if np.linalg.matrix_rank(model.exog) < model.exog.shape[1]:
                n_rank_deficient += 1
                continue
            res = model.fit()
            if res.converged:
                boot_params.append(res.params)
        except Exception:
            continue
    print(f"  (excluded {n_rank_deficient} rank-deficient replicates out of {n_boot} attempted; "
          f"{len(boot_params)} whole-replicate fits retained)")
    boot_df_params = pd.DataFrame(boot_params)
    # Per-term masking, not per-replicate exclusion: a handful of very sparse terms (e.g.
    # Insurance=Yes n=16, Partner occupation=Don't know n=8) are chronically unstable under PSU
    # resampling and would wipe out nearly every replicate if we dropped the whole row for them.
    # Masking only the offending cell preserves the other ~36 well-behaved terms' full 300
    # replicates while still keeping degenerate values out of any single term's percentile CI.
    n_usable_per_term = boot_df_params.notna().sum() - (boot_df_params.abs() > max_abs_coef).sum()
    boot_df_params_masked = boot_df_params.mask(boot_df_params.abs() > max_abs_coef)
    return boot_df_params_masked, n_usable_per_term

boot_dep, n_usable_dep = stratified_cluster_bootstrap(dep, f_dep_m2, n_boot=300)

boot_ci = boot_dep.quantile([0.025, 0.975]).T
boot_ci.columns = ['boot_CI_low_logodds', 'boot_CI_high_logodds']
boot_ci['boot_OR_low'] = np.exp(boot_ci['boot_CI_low_logodds'])
boot_ci['boot_OR_high'] = np.exp(boot_ci['boot_CI_high_logodds'])
boot_ci = boot_ci.drop(columns=['boot_CI_low_logodds', 'boot_CI_high_logodds'])
boot_ci['n_usable_reps'] = n_usable_dep

compare = dep_m2_or.set_index('term')[['OR', 'CI_low', 'CI_high']].join(boot_ci, how='right').drop(index='Intercept', errors='ignore')
print("\nComparison: statsmodels cluster-robust CI vs. stratified cluster-bootstrap CI")
print("(n_usable_reps < ~250 flags a term too sparse for the bootstrap CI to be trustworthy --")
print(" defer to the statsmodels or R/Stata CI for that term instead)")
display(compare.round(3))


  (excluded 1 rank-deficient replicates out of 300 attempted; 299 whole-replicate fits retained)

Comparison: statsmodels cluster-robust CI vs. stratified cluster-bootstrap CI
(n_usable_reps < ~250 flags a term too sparse for the bootstrap CI to be trustworthy --
 defer to the statsmodels or R/Stata CI for that term instead)


,OR,CI_low,CI_high,boot_OR_low,boot_OR_high,n_usable_reps
"C(Q(""Socioeconomic Status""))[T.1]",1.597,0.818,3.118,0.803,3.612,299
"C(Q(""Socioeconomic Status""))[T.2]",2.042,1.084,3.844,1.130,4.185,299
"C(Q(""Socioeconomic Status""))[T.3]",1.787,0.916,3.488,0.941,3.907,299
"C(Q(""Socioeconomic Status""))[T.4]",1.647,0.885,3.068,0.897,3.238,299
"C(Q(""Education""))[T.1]",0.910,0.597,1.387,0.607,1.406,299
"C(Q(""Education""))[T.2]",0.850,0.542,1.335,0.558,1.298,299
"C(Q(""Education""))[T.3]",0.473,0.254,0.881,0.247,0.797,299
"C(Q(""Occupation""))[T.1]",0.923,0.664,1.282,0.672,1.304,299
"C(Q(""Partner occupation""))[T.1]",1.182,0.483,2.890,0.283,2.666,299
"C(Q(""Partner occupation""))[T.3]",0.000,0.000,0.000,NaN,NaN,0


If the `statsmodels` and bootstrap confidence intervals are similar in width and don't
disagree on which side of 1.0 they sit, that's reassurance that the cluster-only sandwich
approximation isn't badly misleading here — but this is still a Python-side cross-check, not a
substitute for the R/Stata replication in Section 11.

## 9. Anxiety models

Identical structure and covariate set as the depression models above.

In [15]:
f_anx_m1  = f'Anxiety_binary ~ C(Q("Socioeconomic Status")) + Burden_num + {covariate_formula}'
f_anx_m2  = f'Anxiety_binary ~ C(Q("Socioeconomic Status")) * Burden_num + {covariate_formula}'
f_anx_m1c = f'Anxiety_binary ~ C(Q("Socioeconomic Status")) + Q("Burden_collapsed") + {covariate_formula}'
f_anx_m3  = f'Anxiety_binary ~ C(Q("Socioeconomic Status")) * Q("Burden_collapsed") + {covariate_formula}'

res_anx_m1,  _  = fit_svy_logit(f_anx_m1,  anx)
res_anx_m2,  w2a = fit_svy_logit(f_anx_m2, anx)
res_anx_m1c, _  = fit_svy_logit(f_anx_m1c, anx)
res_anx_m3,  _  = fit_svy_logit(f_anx_m3,  anx)

print('Model 1 converged:', res_anx_m1.converged, '| Model 2 converged:', res_anx_m2.converged, '| Model 3 converged:', res_anx_m3.converged)

anx_m2_or = or_table(res_anx_m2, 'Anxiety M2')
print("\n=== Anxiety Model 2 (primary): Wealth x Burden odds ratios ===")
display(anx_m2_or.round(3))


Model 1 converged: True | Model 2 converged: True | Model 3 converged: True

=== Anxiety Model 2 (primary): Wealth x Burden odds ratios ===


,model,term,OR,CI_low,CI_high,p
0,Anxiety M2,"C(Q(""Socioeconomic Status""))[T.1]",1.001,0.462,2.169,0.998
1,Anxiety M2,"C(Q(""Socioeconomic Status""))[T.2]",1.302,0.674,2.516,0.433
2,Anxiety M2,"C(Q(""Socioeconomic Status""))[T.3]",1.134,0.559,2.301,0.728
3,Anxiety M2,"C(Q(""Socioeconomic Status""))[T.4]",1.337,0.714,2.506,0.364
4,Anxiety M2,"C(Q(""Education""))[T.1]",0.855,0.543,1.344,0.496
5,Anxiety M2,"C(Q(""Education""))[T.2]",0.939,0.612,1.440,0.772
6,Anxiety M2,"C(Q(""Education""))[T.3]",0.472,0.221,1.008,0.052
7,Anxiety M2,"C(Q(""Occupation""))[T.1]",0.922,0.654,1.300,0.643
8,Anxiety M2,"C(Q(""Partner occupation""))[T.1]",1.336,0.650,2.747,0.431
9,Anxiety M2,"C(Q(""Partner occupation""))[T.3]",1.457,0.161,13.175,0.738


In [16]:
stat_a, df_diff_a, p_a = lr_test(res_anx_m1, res_anx_m2)
print(f"LR test, Anxiety Model 1 vs Model 2 (linear interaction): "
      f"chi2({df_diff_a:.0f}) = {stat_a:.2f}, p = {p_a:.4f}")

stat_a3, df_diff_a3, p_a3 = lr_test(res_anx_m1c, res_anx_m3)
print(f"LR test, Anxiety Model 1c vs Model 3 (collapsed-Burden interaction, correctly nested): "
      f"chi2({df_diff_a3:.0f}) = {stat_a3:.2f}, p = {p_a3:.4f}")


LR test, Anxiety Model 1 vs Model 2 (linear interaction): chi2(4) = 5.74, p = 0.2195
LR test, Anxiety Model 1c vs Model 3 (collapsed-Burden interaction, correctly nested): chi2(8) = 8.43, p = 0.3929


In [17]:
anx_m3_or = or_table(res_anx_m3, 'Anxiety M3')
interaction_rows_a = anx_m3_or[anx_m3_or['term'].str.contains(':')]
print("Anxiety Model 3: Wealth x Burden_collapsed interaction terms (neither Model 2 nor "
      "Model 3 reached significance for anxiety -- shown for completeness/symmetry with "
      "the Depression result above)")
display(interaction_rows_a.round(3))


Anxiety Model 3: Wealth x Burden_collapsed interaction terms (neither Model 2 nor Model 3 reached significance for anxiety -- shown for completeness/symmetry with the Depression result above)


,model,term,OR,CI_low,CI_high,p
34,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.1]:Q(""Burden_co...",0.480,0.129,1.784,0.273
35,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.2]:Q(""Burden_co...",0.414,0.123,1.397,0.155
36,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.3]:Q(""Burden_co...",0.908,0.302,2.734,0.864
37,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.4]:Q(""Burden_co...",1.175,0.410,3.373,0.764
38,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.1]:Q(""Burden_co...",0.000,0.000,0.000,0.000
39,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.2]:Q(""Burden_co...",0.217,0.020,2.325,0.207
40,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.3]:Q(""Burden_co...",0.925,0.103,8.334,0.945
41,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.4]:Q(""Burden_co...",0.411,0.079,2.133,0.290


In [18]:
anx_slopes = simple_slopes(res_anx_m2, wealth_col, 'Burden_num', [0, 1, 2, 3], [1, 2, 3, 4])
print("Anxiety: Wealth OR (vs Richest) at each Cardiometabolic Burden level")
display(anx_slopes.round(3))


Anxiety: Wealth OR (vs Richest) at each Cardiometabolic Burden level


,Wealth (vs Richest),Burden,OR,CI_low,CI_high,p
0,1,0,1.001,0.462,2.169,0.998
1,1,1,0.448,0.164,1.226,0.118
2,1,2,0.201,0.030,1.351,0.099
3,1,3,0.090,0.005,1.648,0.104
4,2,0,1.302,0.674,2.516,0.433
5,2,1,0.593,0.260,1.353,0.214
6,2,2,0.270,0.056,1.307,0.104
7,2,3,0.123,0.011,1.382,0.090
8,3,0,1.134,0.559,2.301,0.728
9,3,1,1.131,0.544,2.351,0.742


In [19]:
boot_anx, n_usable_anx = stratified_cluster_bootstrap(anx, f_anx_m2, n_boot=300)

boot_ci_a = boot_anx.quantile([0.025, 0.975]).T
boot_ci_a.columns = ['boot_CI_low_logodds', 'boot_CI_high_logodds']
boot_ci_a['boot_OR_low'] = np.exp(boot_ci_a['boot_CI_low_logodds'])
boot_ci_a['boot_OR_high'] = np.exp(boot_ci_a['boot_CI_high_logodds'])
boot_ci_a = boot_ci_a.drop(columns=['boot_CI_low_logodds', 'boot_CI_high_logodds'])
boot_ci_a['n_usable_reps'] = n_usable_anx

compare_a = anx_m2_or.set_index('term')[['OR', 'CI_low', 'CI_high']].join(boot_ci_a, how='right').drop(index='Intercept', errors='ignore')
print("\nComparison: statsmodels cluster-robust CI vs. stratified cluster-bootstrap CI (Anxiety)")
display(compare_a.round(3))


  (excluded 1 rank-deficient replicates out of 300 attempted; 299 whole-replicate fits retained)

Comparison: statsmodels cluster-robust CI vs. stratified cluster-bootstrap CI (Anxiety)


,OR,CI_low,CI_high,boot_OR_low,boot_OR_high,n_usable_reps
"C(Q(""Socioeconomic Status""))[T.1]",1.001,0.462,2.169,0.417,2.373,299
"C(Q(""Socioeconomic Status""))[T.2]",1.302,0.674,2.516,0.675,2.718,299
"C(Q(""Socioeconomic Status""))[T.3]",1.134,0.559,2.301,0.556,2.394,299
"C(Q(""Socioeconomic Status""))[T.4]",1.337,0.714,2.506,0.664,2.552,299
"C(Q(""Education""))[T.1]",0.855,0.543,1.344,0.501,1.323,299
"C(Q(""Education""))[T.2]",0.939,0.612,1.440,0.615,1.468,299
"C(Q(""Education""))[T.3]",0.472,0.221,1.008,0.201,0.945,299
"C(Q(""Occupation""))[T.1]",0.922,0.654,1.300,0.643,1.250,299
"C(Q(""Partner occupation""))[T.1]",1.336,0.650,2.747,0.495,2.483,299
"C(Q(""Partner occupation""))[T.3]",1.457,0.161,13.175,0.608,10.973,190


## 10. Combined summary: Wealth × Burden interaction, both outcomes

In [20]:
summary_rows = []
for outcome, res_m1, res_m2, lr_stat, lr_df, lr_p in [
    ('Depression', res_dep_m1, res_dep_m2, stat, df_diff, p),
    ('Anxiety', res_anx_m1, res_anx_m2, stat_a, df_diff_a, p_a),
]:
    summary_rows.append({
        'Outcome': outcome,
        'N': int(res_m2.nobs),
        'Model 1 AIC': round(res_m1.aic, 1),
        'Model 2 AIC': round(res_m2.aic, 1),
        'LR chi2': round(lr_stat, 2),
        'df': int(lr_df),
        'LR p-value': round(lr_p, 4),
        'Interaction significant (α=.05)': 'Yes' if lr_p < 0.05 else 'No',
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)


,Outcome,N,Model 1 AIC,Model 2 AIC,LR chi2,df,LR p-value,Interaction significant (α=.05)
0,Depression,4887,1878.2,1879.9,6.25,4,0.1816,No
1,Anxiety,4887,1806.4,1808.7,5.74,4,0.2195,No


## 11. Exact replication in R / Stata (for final reported standard errors)

Point estimates (odds ratios) from Section 6–9 will match R/Stata exactly, since they don't
depend on the variance estimator. Standard errors, confidence intervals, and p-values should be
taken from one of these for the submitted manuscript, not from the Python cluster-robust or
bootstrap estimates above, which are cross-checks.

**R (`survey` package):**
```r
library(survey)
dep <- read.csv("resources/depression_dataset.csv")
dep$SES <- relevel(factor(dep$Socioeconomic.Status), ref = "5")

des <- svydesign(
  id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight,
  data = dep, nest = TRUE
)

m1 <- svyglm(Depression_binary ~ SES + Burden_num + Education + Occupation + ...,
             design = des, family = quasibinomial())
m2 <- svyglm(Depression_binary ~ SES * Burden_num + Education + Occupation + ...,
             design = des, family = quasibinomial())
anova(m1, m2, method = "LRT")   # nested-model test, design-based
```

**Stata:**
```stata
import delimited "resources/depression_dataset.csv", clear
svyset psu [pw=sampling_weight], strata(stratum)

svy: logit depression_binary i.ses##c.burden_num i.education i.occupation ...
```

The `nest = TRUE` (R) / `svyset ..., strata()` (Stata) options are exactly the piece
`statsmodels` cannot do — that's the whole reason this section exists.

## 12. Interpretation

*(Fill in / adjust the bracketed values once the R or Stata replication above confirms the
standard errors — the odds ratios themselves are already final.)*

**Overall interaction test:** the likelihood-ratio test comparing Model 1 to Model 2
(Section 6/9 output) tells you whether cardiometabolic burden significantly modifies the
wealth–mental-health association *at all*, for each outcome, before looking at any individual
coefficient. Interpret individual simple-slope ORs (Section 7) only if this omnibus test is
significant — otherwise, individual "significant-looking" simple slopes may just be
multiple-comparison noise.

**How to read the simple-slopes table (Section 7):** each row is the odds of probable
depression/anxiety for a given wealth category *relative to the Richest group*, holding
covariates fixed, evaluated at a specific cardiometabolic-burden level. If the OR for e.g.
`Poorest` grows (moves further from 1) as `Burden` increases from 0 → 3, that is the effect-
modification story: wealth-related disparities in mental health widen among women carrying more
cardiometabolic burden. If the OR is roughly flat across burden levels, there's no meaningful
effect modification for that wealth category even if the raw wealth main effect is significant.

**Model 3 (collapsed-Burden) sensitivity check:** if Model 3's interaction pattern points the
same direction as Model 2's linear-interaction slopes, that's evidence the linearity assumption
in the primary model isn't driving the result. If they disagree, the linear-Burden assumption
needs more scrutiny (e.g., a quadratic term, or reporting Model 3 as primary instead).

**Before writing this up as final:**
1. Re-run Section 6–9 with R/Stata design-based SEs and update the CIs/p-values quoted above.
2. Report the Model 1→2 LR test as the primary evidence for/against effect modification, not
   individual interaction-term p-values (which are individually underpowered given the sparse
   cells discussed in Section 4).
3. State the PHQ-9/GAD-7 ≥10 cutoff and its citation explicitly in Methods (Section 2) — this is
   a modeling choice, and a reviewer may ask for a sensitivity analysis at a different cutoff.
4. Confirm the `Financial Decision-Making` / DAG open question (flagged in `docs/to_fix.md`)
   before finalizing the covariate set, since adding it would change every model in this
   notebook.